In [ ]:
import pandas as pd
from collections import defaultdict

# =============================
# Load data
# =============================
df = pd.read_excel("input.xlsx")

df["Plan"] = pd.to_numeric(df["Plan"], errors="coerce").fillna(0)
df["Cycle Time"] = pd.to_numeric(df["Cycle Time"], errors="coerce").fillna(0)

# =============================
# Constants
# =============================
USABLE_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 3960  # minutes (22 hrs × 3 days)

# =============================
# Tracking structures
# =============================
machine_load = {m: 0 for m in USABLE_MACHINES}
machine_plan = defaultdict(list)
rejection_log = []

# =============================
# Allocation logic
# =============================
for _, row in df.iterrows():

    child = row["Child Part"]
    required_qty = row["Plan"]
    cycle_time = row["Cycle Time"]

    if required_qty <= 0 or cycle_time <= 0:
        continue

    required_time = required_qty * cycle_time

    vertical_machines = str(row["Vertical Machines"]).split(",")
    vertical_machines = [m.strip() for m in vertical_machines]

    eligible = [m for m in vertical_machines if m in USABLE_MACHINES]

    if not eligible:
        rejection_log.append((child, "No eligible 120T machine"))
        continue

    remaining_time = required_time

    eligible.sort(key=lambda m: machine_load[m])

    for m in eligible:
        if remaining_time <= 0:
            break

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc_time = min(available, remaining_time)
        alloc_qty = alloc_time / cycle_time

        machine_load[m] += alloc_time
        remaining_time -= alloc_time

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(alloc_qty, 2),
            "Time Used (min)": round(alloc_time, 2)
        })

    if remaining_time > 0:
        rejection_log.append((
            child,
            f"Shortfall qty {round(remaining_time / cycle_time, 2)}"
        ))

# =============================
# DISPLAY RESULTS
# =============================

print("\n================ MACHINE-WISE PLAN ================\n")
for m, plans in machine_plan.items():
    print(f"🔧 {m}")
    display(pd.DataFrame(plans))
    print("-" * 60)

print("\n================ MACHINE LOAD SUMMARY ================\n")
load_df = pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in USABLE_MACHINES
])
display(load_df)

print("\n================ REJECTIONS / SHORTFALLS ================\n")
if rejection_log:
    display(pd.DataFrame(rejection_log, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections. All plans feasible.")


In [ ]:
import pandas as pd
from collections import defaultdict

# =============================
# Load data
# =============================
df = pd.read_excel("input.xlsx")

df["Plan"] = pd.to_numeric(df["Plan"], errors="coerce").fillna(0)
df["Cycle Time"] = pd.to_numeric(df["Cycle Time"], errors="coerce").fillna(0)

# =============================
# EXPLICIT MACHINE ALLOW LIST
# =============================
ALLOWED_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

MACHINE_CAPACITY = 3960  # minutes (22 hrs × 3 days)

# =============================
# Tracking
# =============================
machine_load = {m: 0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
rejection_log = []

# =============================
# Allocation
# =============================
for _, row in df.iterrows():

    child = row["Child Part"]
    required_qty = row["Plan"]
    cycle_time = row["Cycle Time"]

    if required_qty <= 0 or cycle_time <= 0:
        continue

    required_time = required_qty * cycle_time

    vertical_machines = [
        m.strip() for m in str(row["Vertical Machines"]).split(",")
    ]

    # STRICT FILTER — name-based only
    eligible = [m for m in vertical_machines if m in ALLOWED_MACHINES]

    if not eligible:
        rejection_log.append((child, "No allowed machine"))
        continue

    remaining_time = required_time

    # Load balancing
    eligible.sort(key=lambda m: machine_load[m])

    for m in eligible:
        if remaining_time <= 0:
            break

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc_time = min(available, remaining_time)
        alloc_qty = alloc_time / cycle_time

        machine_load[m] += alloc_time
        remaining_time -= alloc_time

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(alloc_qty, 2),
            "Time Used (min)": round(alloc_time, 2)
        })

    if remaining_time > 0:
        rejection_log.append((
            child,
            f"Shortfall qty {round(remaining_time / cycle_time, 2)}"
        ))

# =============================
# DISPLAY RESULTS
# =============================

print("\n========== MACHINE-WISE PLAN ==========\n")
for m in sorted(ALLOWED_MACHINES):
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No assigned parts")
    print("-" * 50)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in sorted(ALLOWED_MACHINES)
]))

print("\n========== REJECTIONS ==========\n")
if rejection_log:
    display(pd.DataFrame(rejection_log, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections")
